# 02 - Action：实验设计

> **STAR法则的第二步**：设计科学的实验方案。

---

## 📋 学习目标

完成本notebook后，你将能够：
1. 计算实验所需样本量
2. 确定实验周期
3. 设计随机化方案
4. 设定统计参数（α、β、MDE）

---

## 🎯 实验设计要素

### 1. 统计参数设定

**TODO 1**：设定实验的统计参数，并解释选择的理由。


In [ ]:
# 统计参数
alpha = 0.05      # 显著性水平
power = 0.80      # 统计功效（1-β）
mde_relative = 0.15  # 相对最小可检测效应（15%）

# 历史数据（用于样本量计算）
historical_mean = 1800  # 停留时长均值（秒）
historical_std = 1200   # 停留时长标准差（秒）

print(f"统计参数设定：")
print(f"  显著性水平 α = {alpha}")
print(f"  统计功效 1-β = {power}")
print(f"  相对MDE = {mde_relative*100}%")
print(f"  历史均值 = {historical_mean}秒")
print(f"  历史标准差 = {historical_std}秒")

# 解释你的选择
reason = """
解释为什么这样选择：
1. α=0.05：...
2. power=0.80：...
3. MDE=15%：...
"""
print(f"
选择理由：
{reason}")



### 2. 样本量计算

**TODO 2**：计算实验所需样本量。


In [ ]:
import numpy as np
from scipy import stats

def sample_size_continuous(mean, std, mde_relative, alpha=0.05, power=0.8):
    """
    计算连续型指标的样本量
    
    参数:
        mean: 对照组均值
        std: 对照组标准差
        mde_relative: 相对最小可检测效应
        alpha: 显著性水平
        power: 统计功效
    
    返回:
        每组所需样本量
    """
    # 完成样本量计算公式
    # 使用z检验的样本量公式
    # n = 2 * ((z_α/2 + z_β) * σ / δ)²
    
    mde_absolute = mean * mde_relative  # 绝对MDE
    
    # 计算z值
    z_alpha = stats.norm.ppf(1 - alpha/2)
    z_beta = stats.norm.ppf(power)
    
    # 计算样本量
    n = 2 * ((z_alpha + z_beta) * std / mde_absolute) ** 2
    
    return int(np.ceil(n))

# 计算样本量
n_per_group = sample_size_continuous(historical_mean, historical_std, mde_relative, alpha, power)
total_n = n_per_group * 2

print(f"每组需要样本量: {n_per_group}")
print(f"总样本量: {total_n}")



### 3. 实验周期计算

**TODO 3**：根据样本量和DAU计算实验周期。


In [ ]:
# 假设
dau = 5_000_000  # 日活跃用户
traffic_ratio = 0.10  # 可用于实验的流量比例（10%）

# 计算实验周期
daily_experiment_users = dau * traffic_ratio
days_needed = int(np.ceil(total_n / daily_experiment_users))

print(f"DAU: {dau:,}")
print(f"实验流量比例: {traffic_ratio*100}%")
print(f"每日实验用户数: {daily_experiment_users:,}")
print(f"所需实验天数: {days_needed}天")

# 思考：如果产品团队说"只能跑1周"，你怎么办？
print("\n" + "="*60)
print("业务干扰场景：产品团队希望1周出结论")
print("="*60)

response = """
应对策略：
1. 说明统计功效不足的风险，1周样本量不够，假阴性率高
2. 提出折中方案：提高MDE到20%或提高alpha到0.1，但需明确告知业务方风险
3. 建议分阶段：先跑1周看趋势，不提前下结论，继续跑满2周
"""
print(response)



---

## 🎲 随机化方案

### 随机化单元

**TODO 4**：确定随机化单元，并解释为什么。


In [ ]:
randomization_unit = "user_id"  # 填写：user_id / session_id / device_id / ...

reason = """
解释为什么选择这个随机化单元：
1. 用户级别的实验体验一致，避免同一用户在不同设备上看到不同推荐
2. 停留时长和留存都是用户级别的指标，user_id作为随机化单元最匹配
"""

print(f"随机化单元: {randomization_unit}")
print(f"选择理由: \n{reason}")



### 随机化方法

**TODO 5**：实现用户分配函数。


In [ ]:
import hashlib

def assign_group(user_id, salt="experiment_2024_01", num_buckets=1000):
    """
    基于用户ID的确定性随机化
    确保同一用户始终分配到同一组
    
    参数:
        user_id: 用户ID
        salt: 实验盐值，确保不同实验独立
        num_buckets: 分桶数
    
    返回:
        'control' 或 'treatment'
    """
    # 实现随机化逻辑
    hash_input = f"{user_id}_{salt}"
    hash_value = int(hashlib.md5(hash_input.encode()).hexdigest(), 16)
    bucket = hash_value % num_buckets
    return "control" if bucket < num_buckets // 2 else "treatment"

# 测试
test_users = [f"U_{i}" for i in range(10)]
for user in test_users:
    group = assign_group(user)
    print(f"{user} -> {group}")

# 验证一致性
print("\n验证一致性（同一用户多次分配结果相同）：")
user = "U_12345"
groups = [assign_group(user) for _ in range(5)]
print(f"{user} 分配5次: {groups}")
print(f"是否一致: {len(set(groups)) == 1}")



---

## 🧪 AA检验设计

**TODO 6**：设计AA检验方案。


In [ ]:
aa_test_design = """
AA检验方案：

1. AA检验的目的：验证随机化是否成功，两组在实验前是否平衡
2. AA检验的指标：historical_dwell_time（历史停留时长）
3. AA检验的判断标准：t检验p值 > 0.05，认为两组无显著差异
4. 如果AA检验不通过，怎么办：检查随机化逻辑、数据质量，必要时重新分组
"""

print(aa_test_design)

# 思考：如果不做AA检验，会有什么后果？
print("\n" + "="*60)
print("失败路径：跳过AA检验的后果")
print("="*60)

consequence = """
跳过AA检验的后果：

场景：假设对照组历史停留时长均值1800秒，实验组历史停留时长均值2000秒
（分组时由于随机化问题，实验组天然偏高）

后果：
1. 实验组天然优势200秒，即使新算法无效也会显得有效
2. 假阳性率大幅上升，可能上线无效甚至有害的算法
3. 后续无法归因，业务决策基于错误数据
"""

print(consequence)



---

## 📋 实验设计文档

**TODO 7**：整理完整的实验设计文档。


In [ ]:
experiment_design = {
    "实验名称": "",
    "实验目的": "",
    "主要指标（OEC）": "",
    "Guardrail指标": [],
    "统计参数": {
        "alpha": alpha,
        "power": power,
        "mde_relative": mde_relative
    },
    "样本量": {
        "每组": n_per_group,
        "总计": total_n
    },
    "实验周期": f"{days_needed}天",
    "随机化": {
        "单元": "user_id",
        "方法": "基于MD5 hash的确定性分桶",
        "流量比例": traffic_ratio
    },
    "AA检验": {
        "指标": ["historical_dwell_time"],
        "判断标准": "t检验p > 0.05"
    },
    "分析计划": {
        "主要分析方法": "t检验（unequal var）",
        "多重检验校正": "Bonferroni + FDR",
        "敏感性分析": "不同MDE和alpha下的样本量和显著性"
    }
}

import json
print(json.dumps(experiment_design, indent=2, ensure_ascii=False))



---

## 📝 本节总结

完成本节，你应该已经：

- [ ] 设定统计参数并解释理由
- [ ] 计算样本量
- [ ] 确定实验周期
- [ ] 设计随机化方案
- [ ] 设计AA检验方案
- [ ] 整理完整的实验设计文档

**下一步**：进入 `03-action-execution.ipynb`，执行实验并生成数据。

---

> 💡 **关键洞察**：
> 实验设计阶段决定了实验的上限。
> 设计不严谨，分析再漂亮也救不回来。
